In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

In [2]:
Genomic_Feature_Matrix_Path = r"C:\Users\tharu\Desktop\Starlight\FYP\Cross-Disorder_Multi-Label_Classification_Using_Transformer_and_Graph-Based_Fusion_for_Dyslexia_ADHD_and_Autism\Implementation\Genomic Data\Preprocessing Pipelines\genetic_feature_matrix.csv"

Genomic_Feature_Matrix = pd.read_csv(Genomic_Feature_Matrix_Path)

In [3]:
Genomic_Feature_Matrix

,Unnamed: 0,ABC Transporter Disorders R-HSA-5619084,ABC Transporters In Lipid Homeostasis R-HSA-1369062,ABC-family Proteins Mediated Transport R-HSA-382556,ADORA2B Mediated Anti-Inflammatory Cytokine Production R-HSA-9660821,ADP Signaling Thru P2Y Purinoceptor 1 R-HSA-418592,AKT Phosphorylates Targets In Nucleus R-HSA-198693,ALK Mutants Bind TKIs R-HSA-9700645,APC Truncation Mutants Have Impaired AXIN Binding R-HSA-5467337,APC-Cdc20 Mediated Degradation Of Nek2A R-HSA-179409,...,p75NTR Signals Via NF-kB R-HSA-193639,rRNA Modification In Mitochondrion R-HSA-6793080,rRNA Modification In Nucleus And Cytosol R-HSA-6790901,rRNA Processing In Mitochondrion R-HSA-8868766,rRNA Processing In Nucleus And Cytosol R-HSA-8868773,rRNA Processing R-HSA-72312,tRNA Aminoacylation R-HSA-379724,tRNA Processing In Nucleus R-HSA-6784531,tRNA Processing R-HSA-72306,trans-Golgi Network Vesicle Budding R-HSA-199992
0,ADHD,0.152650,1.019885,-0.039933,0.646598,-1.070600,-1.070600,-1.070600,-1.070600,0.875863,...,1.127755,-1.070600,-1.070600,-1.070600,0.802932,0.781559,-1.070600,0.468161,-0.058058,-1.070600
1,ASD,-0.502054,-0.502054,-0.502054,1.202903,-0.502054,-0.502054,-0.502054,-0.502054,-0.502054,...,-0.502054,-0.502054,-0.502054,-0.502054,1.932551,1.932551,-0.502054,-0.502054,-0.502054,-0.502054
2,Dyslexia,0.246506,-1.325420,-0.046490,-0.386510,0.877563,0.918492,0.918492,0.918492,-1.325420,...,-1.325420,0.918492,0.430758,0.918492,-0.811817,0.111672,0.622428,-1.325420,-1.325420,0.918492


In [4]:
print(Genomic_Feature_Matrix.shape)  # (3, 974)

(3, 1029)


#### Standardize 

In [5]:
X = Genomic_Feature_Matrix.iloc[:, 1:].values  # drop the first column
y = Genomic_Feature_Matrix.iloc[:, 0].values   # optional: keep labels

scaler = StandardScaler()
Genomic_Feature_Matrix_Scaled = scaler.fit_transform(X)  # shape (3, 974)

print("Scaled Shape: ", Genomic_Feature_Matrix_Scaled.shape)
Genomic_Feature_Matrix_Scaled

Scaled Shape:  (3, 1028)


array([[ 0.56146938,  1.32686861,  0.72220744, ...,  1.25676676,
         1.08643515, -1.01917508],
       [-1.40481836, -0.23968347, -1.41410529, ..., -0.06677589,
         0.24083855, -0.33950704],
       [ 0.84334898, -1.08718514,  0.69189785, ..., -1.18999087,
        -1.3272737 ,  1.35868212]], shape=(3, 1028))

#### Remove 0 valued columns

In [6]:
nonzero_cols = np.where(Genomic_Feature_Matrix_Scaled.sum(axis=0) != 0)[0]
Genomic_Feature_Matrix_Scaled = Genomic_Feature_Matrix_Scaled[:, nonzero_cols]

Genomic_Feature_Matrix_Scaled

array([[ 0.56146938,  0.72220744,  0.24134548, ..., -0.21326404,
         1.08643515, -1.01917508],
       [-1.40481836, -1.41410529,  1.08610572, ...,  1.31737098,
         0.24083855, -0.33950704],
       [ 0.84334898,  0.69189785, -1.32745121, ..., -1.10410694,
        -1.3272737 ,  1.35868212]], shape=(3, 727))

#### Convert to PyTorch tensor

In [7]:
X_genomic = torch.tensor(Genomic_Feature_Matrix_Scaled, dtype=torch.float32)  

#### Define the MLP encoder

In [8]:
class GenomicEncoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embedding_dim)
        )
    
    def forward(self, x):
        return self.encoder(x)

#### Initialize encoder and get embeddings

In [9]:
input_dim = X_genomic.shape[1]
embedding_dim = 128

encoder = GenomicEncoder(input_dim=input_dim, embedding_dim=embedding_dim)

with torch.no_grad():  # no training needed if only projecting
    embeddings = encoder(X_genomic)
    
print(embeddings.shape)  # (3, 128)

torch.Size([3, 128])


#### Save Embeddings and Labels

In [10]:
# Convert PyTorch tensor to NumPy
embeddings_np = embeddings.numpy()  # shape (3, 128)

# Save to .npy file
np.save("genomic_embeddings.npy", embeddings_np)

print("Embeddings saved! Shape:", embeddings_np.shape)

Embeddings saved! Shape: (3, 128)


In [11]:
loaded_embeddings = np.load("genomic_embeddings.npy")
print(loaded_embeddings.shape)  # should be (3, 128)

(3, 128)


In [12]:
loaded_embeddings

array([[ 6.44701302e-01,  1.53771013e-01, -4.77894098e-02,
        -3.44791263e-03,  1.01069041e-01,  2.27863863e-02,
        -4.73288208e-01, -7.78862685e-02,  5.80368042e-01,
         5.97824395e-01, -1.41183525e-01, -4.60075229e-01,
        -5.72815478e-01, -1.45736098e-01,  3.64050448e-01,
         3.74572039e-01,  1.43331513e-01,  1.27452508e-01,
        -4.05932575e-01, -3.11964542e-01, -1.34263873e-01,
        -2.75630563e-01, -2.94510990e-01,  2.31029361e-01,
        -4.39043790e-02, -7.55734742e-02,  1.29398674e-01,
         2.52433158e-02,  3.46846700e-01, -1.45191088e-01,
         1.24515243e-01,  2.06776947e-01, -1.86371595e-01,
        -3.88647199e-01,  8.16153511e-02,  6.68298677e-02,
         4.50294688e-02,  2.32502565e-01, -8.20121244e-02,
        -3.44887346e-01,  1.06871903e-01, -3.57761174e-01,
        -2.27984458e-01,  1.14207834e-01,  5.95492572e-02,
         8.49704266e-01, -3.90236646e-01,  1.01825558e-01,
        -1.05569042e-01, -1.97137713e-01,  2.53705740e-0

In [13]:
labels = Genomic_Feature_Matrix.iloc[:, 0].values  # ['ADHD', 'ASD', 'Dyslexia']
np.save("genomic_labels.npy", labels)

#### cosine similarity matrix (To check if the embeddings have been well extracted)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings.numpy())
print(similarity_matrix)

[[1.         0.24067846 0.05279616]
 [0.24067846 0.9999996  0.17215545]
 [0.05279616 0.17215545 0.9999998 ]]


After generating embeddings for ADHD, ASD, and Dyslexia using the trained model, we computed pairwise cosine similarities to quantify how similar the learned representations are across disorders. 

The resulting similarity matrix shows high values along the diagonal (self-similarity = 1) and lower off-diagonal values, reflecting cross-disorder relationships. 

Specifically, ADHD and ASD embeddings show moderate similarity (≈0.24), indicating partially shared genomic or phenotypic patterns, whereas Dyslexia exhibits low similarity with both ADHD (≈0.05) and ASD (≈0.17), highlighting its distinctiveness. 

These results suggest that the embeddings effectively capture both shared and disorder-specific latent features, providing a biologically meaningful representation suitable for downstream analysis such as clustering, visualization, or linear probing.